# 🏠 House Prices — Feature Engineering

**Goal:** Create new meaningful features, handle skewness, encode categoricals, and log-transform the target variable.

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import skew

pd.set_option("display.max_columns", None)


## 1. Load Clean Data

In [2]:
train_df = pd.read_csv("../data/train_clean.csv")
test_df  = pd.read_csv("../data/test_clean.csv")

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)


Train shape: (1458, 80)
Test shape : (1459, 79)


## 2. Separate & Log-Transform Target Variable

⚠️ **Critical:** Kaggle evaluates using **RMSLE**. RMSLE = RMSE on log-transformed values. So we must `log1p` the target for training.

In [3]:
y_raw = train_df["SalePrice"]
y_log = np.log1p(y_raw)   # ← log-transformed target for training

train_df = train_df.drop("SalePrice", axis=1)

print("Target (raw)  — mean:", round(y_raw.mean(), 2), "| skew:", round(y_raw.skew(), 4))
print("Target (log1p)— mean:", round(y_log.mean(), 2), "| skew:", round(y_log.skew(), 4))


Target (raw)  — mean: 180932.92 | skew: 1.8813
Target (log1p)— mean: 12.02 | skew: 0.1216


## 3. Combine Train & Test for Feature Engineering

Combining prevents train/test column mismatch after one-hot encoding.

In [4]:
n_train = len(train_df)

combined = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

print("Combined shape:", combined.shape)


Combined shape: (2917, 79)


## 4. Create New Features

In [5]:
# Total square footage (all floors + basement)
combined["TotalSF"] = (
    combined["TotalBsmtSF"] +
    combined["1stFlrSF"] +
    combined["2ndFlrSF"]
)


In [6]:
# Total bathrooms (half bath = 0.5)
combined["TotalBathrooms"] = (
    combined["FullBath"] +
    (0.5 * combined["HalfBath"]) +
    combined["BsmtFullBath"] +
    (0.5 * combined["BsmtHalfBath"])
)


In [7]:
# House age at time of sale (clip negative to 0 — prevents log1p NaN)
combined["HouseAge"] = np.maximum(
    combined["YrSold"] - combined["YearBuilt"], 0
)


In [8]:
# Years since remodel at time of sale
combined["RemodeledAge"] = np.maximum(
    combined["YrSold"] - combined["YearRemodAdd"], 0
)


In [9]:
# Was the house ever remodeled?
combined["Remodeled"] = (
    combined["YearRemodAdd"] != combined["YearBuilt"]
).astype(int)


In [10]:
# Total porch area
combined["TotalPorchSF"] = (
    combined["OpenPorchSF"] +
    combined["EnclosedPorch"] +
    combined["3SsnPorch"] +
    combined["ScreenPorch"]
)


In [11]:
# Binary flags
combined["HasGarage"]   = (combined["GarageArea"]   > 0).astype(int)
combined["HasBasement"] = (combined["TotalBsmtSF"]  > 0).astype(int)
combined["HasPool"]     = (combined["PoolArea"]      > 0).astype(int)
combined["Has2ndFloor"] = (combined["2ndFlrSF"]      > 0).astype(int)


In [12]:
# Quality × Area interaction — very strong predictor
combined["QualSF"] = combined["OverallQual"] * combined["TotalSF"]


In [13]:
# Verify new features
new_features = [
    "TotalSF", "TotalBathrooms", "HouseAge", "RemodeledAge",
    "Remodeled", "TotalPorchSF", "HasGarage", "HasBasement",
    "HasPool", "Has2ndFloor", "QualSF"
]
combined[new_features].head()


,TotalSF,TotalBathrooms,HouseAge,RemodeledAge,Remodeled,TotalPorchSF,HasGarage,HasBasement,HasPool,Has2ndFloor,QualSF
0,2566.0,3.5,5,5,0,61,1,1,0,1,17962.0
1,2524.0,2.5,31,31,0,0,1,1,0,0,15144.0
2,2706.0,3.5,7,6,1,42,1,1,0,1,18942.0
3,2473.0,2.0,91,36,1,307,1,1,0,1,17311.0
4,3343.0,3.5,8,8,0,84,1,1,0,1,26744.0


## 5. Fix Skewed Numeric Features

Skewed features hurt linear models. Apply `log1p` to features with |skewness| > 0.75.

In [14]:
numeric_features = combined.select_dtypes(include=[np.number]).columns.tolist()

skewness = combined[numeric_features].apply(lambda x: skew(x.dropna()))
skewed   = skewness[abs(skewness) > 0.75].index

print(f"Skewed features found: {len(skewed)}")
print(list(skewed))


Skewed features found: 27
['MSSubClass', 'LotFrontage', 'LotArea', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtHalfBath', 'KitchenAbvGr', 'GarageYrBlt', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'TotalSF', 'TotalPorchSF', 'HasGarage', 'HasBasement', 'HasPool', 'QualSF']


In [15]:
# Clip to 0 before log1p to avoid NaN on negative values
combined[skewed] = combined[skewed].clip(lower=0)
combined[skewed] = np.log1p(combined[skewed])

print("log1p transform applied to skewed features.")
combined[skewed].head()


log1p transform applied to skewed features.


,MSSubClass,LotFrontage,LotArea,MasVnrArea,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtHalfBath,KitchenAbvGr,GarageYrBlt,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,TotalSF,TotalPorchSF,HasGarage,HasBasement,HasPool,QualSF
0,4.110874,4.189655,9.042040,5.283204,6.561031,0.0,5.017280,6.753438,6.751101,0.0,7.444833,0.000000,0.693147,7.602900,0.000000,4.127134,0.000000,0.0,0.0,0.0,0.0,7.850493,4.127134,0.693147,0.693147,0.0,9.796069
1,3.044522,4.394449,9.169623,0.000000,6.886532,0.0,5.652489,7.141245,0.000000,0.0,7.141245,0.693147,0.693147,7.589336,5.700444,0.000000,0.000000,0.0,0.0,0.0,0.0,7.833996,0.000000,0.693147,0.693147,0.0,9.625426
2,4.110874,4.234107,9.328212,5.093750,6.188264,0.0,6.075346,6.825460,6.765039,0.0,7.488294,0.000000,0.693147,7.601902,0.000000,3.761200,0.000000,0.0,0.0,0.0,0.0,7.903596,3.761200,0.693147,0.693147,0.0,9.849190
3,4.262680,4.110874,9.164401,0.000000,5.379897,0.0,6.293419,6.869014,6.629363,0.0,7.448916,0.000000,0.693147,7.600402,0.000000,3.583519,5.609472,0.0,0.0,0.0,0.0,7.813592,5.730100,0.693147,0.693147,0.0,9.759155
4,4.110874,4.442651,9.565284,5.860786,6.486161,0.0,6.196444,7.044033,6.960348,0.0,7.695758,0.000000,0.693147,7.601402,5.262690,4.442651,0.000000,0.0,0.0,0.0,0.0,8.114923,4.442651,0.693147,0.693147,0.0,10.194103


## 6. One-Hot Encode Categoricals

In [16]:
combined = pd.get_dummies(combined, dtype=int)  # dtype=int → avoids bool dtype issues

print("Combined shape after encoding:", combined.shape)


Combined shape after encoding: (2917, 297)


## 7. Split Back into Train & Test

In [17]:
train_final = combined.iloc[:n_train, :].copy()
test_final  = combined.iloc[n_train:, :].copy()

print("train_final shape:", train_final.shape)
print("test_final shape :", test_final.shape)
print("y_log shape      :", y_log.shape)


train_final shape: (1458, 297)
test_final shape : (1459, 297)
y_log shape      : (1458,)


## 8. Save Everything

In [18]:
# Save log-transformed target
y_log.to_csv("../data/target.csv", index=False)

# Save feature matrices
train_final.to_csv("../data/train_final.csv", index=False)
test_final.to_csv("../data/test_final.csv",   index=False)

print("✅ Feature Engineering Completed Successfully!")
print(f"   target.csv      → {y_log.shape}")
print(f"   train_final.csv → {train_final.shape}")
print(f"   test_final.csv  → {test_final.shape}")


✅ Feature Engineering Completed Successfully!
   target.csv      → (1458,)
   train_final.csv → (1458, 297)
   test_final.csv  → (1459, 297)
